# Task 2: Text Chunking, Embedding, and Vector Store Indexing

Converts cleaned complaint narratives into a FAISS vector store for semantic search.

**Steps:**
1. Load `cleaned_complaints.csv`
2. Chunk each narrative with `RecursiveCharacterTextSplitter`
3. Embed chunks using `sentence-transformers/all-MiniLM-L6-v2`
4. Index into FAISS with metadata (`complaint_id`, `product`, `chunk_index`)
5. Save index to `data/faiss_index/`
6. Smoke-test with a sample query

## 0. Imports & Paths

In [1]:
import os
import sys
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

BASE_DIR  = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR  = os.path.join(BASE_DIR, "data")
INPUT_CSV = os.path.join(DATA_DIR, "cleaned_complaints.csv")
INDEX_DIR = os.path.join(DATA_DIR, "faiss_index")

print("Input CSV :", INPUT_CSV)
print("Index dir :", INDEX_DIR)

/home/hp/Desktop/chatbot/Customer-Feedback-Chabot/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Input CSV : /home/hp/Desktop/chatbot/Customer-Feedback-Chabot/data/cleaned_complaints.csv
Index dir : /home/hp/Desktop/chatbot/Customer-Feedback-Chabot/data/faiss_index


## 1. Load Data

In [2]:
df = pd.read_csv(INPUT_CSV)
before = len(df)

# Keep only rows that have a cleaned narrative to embed
df = df[df["cleaned_narrative"].notna() & (df["cleaned_narrative"].str.strip() != "")]

print(f"Total rows    : {before}")
print(f"Rows with text: {len(df)}")
print(f"Dropped       : {before - len(df)}")
df[["Complaint ID", "Product", "cleaned_narrative"]].head(3)

Total rows    : 82164
Rows with text: 82163
Dropped       : 1


,Complaint ID,Product,cleaned_narrative
0,14069121,Credit card,a card was opened under my name by a fraudster...
1,14047085,Credit card,dear cfpb i have a secured credit card with ci...
2,14040217,Credit card,i have a citi rewards cards the credit balance...


## 2. Text Chunking

**Strategy:** `RecursiveCharacterTextSplitter`
- `chunk_size = 500` chars — preserves sentence context while keeping vectors focused
- `chunk_overlap = 50` chars — prevents meaning loss at chunk boundaries
- Separators try sentence endings first (`. `, `! `, `? `), then newlines, then words

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=[". ", "! ", "? ", "\n", " ", ""],
)

documents = []
for _, row in df.iterrows():
    chunks = splitter.split_text(str(row["cleaned_narrative"]))
    for i, chunk in enumerate(chunks):
        documents.append(
            Document(
                page_content=chunk,
                metadata={
                    "complaint_id": str(row["Complaint ID"]),
                    "product":      str(row["Product"]),
                    "chunk_index":  i,
                },
            )
        )

print(f"Complaints        : {len(df)}")
print(f"Total chunks      : {len(documents)}")
print(f"Avg chunks/complaint: {len(documents)/len(df):.1f}")

Complaints        : 82163
Total chunks      : 222818
Avg chunks/complaint: 2.7


In [4]:
# Inspect a sample chunk
sample = documents[0]
print("Content :", sample.page_content)
print("Metadata:", sample.metadata)

Content : a card was opened under my name by a fraudster i received a notice from that an account was just opened under my name i reached out to to state that this activity was unauthorized and not me confirmed this was fraudulent and immediately closed the card however they have failed to remove this from the three credit agencies and this fraud is now impacting my credit score based on a hard credit pull done by that was done by a fraudster
Metadata: {'complaint_id': '14069121', 'product': 'Credit card', 'chunk_index': 0}


## 3. Embedding Model

**Model:** `sentence-transformers/all-MiniLM-L6-v2`
- Lightweight (~80 MB), runs fully on CPU with no API key
- Produces 384-dim vectors optimised for semantic similarity
- Strong benchmark scores on short-to-medium English text — well-suited for complaint narratives

In [5]:
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("Model loaded.")

Loading embedding model...


/tmp/ipykernel_71463/116159950.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1407.16it/s]


Model loaded.


## 4. Build & Save FAISS Index

In [6]:
print(f"Embedding {len(documents)} chunks — this may take a few minutes...")
vector_store = FAISS.from_documents(documents, embeddings)
print("Embedding complete.")

Embedding 222818 chunks — this may take a few minutes...
Embedding complete.


In [7]:
os.makedirs(INDEX_DIR, exist_ok=True)
vector_store.save_local(INDEX_DIR)

saved_files = os.listdir(INDEX_DIR)
print(f"Index saved to: {INDEX_DIR}")
for f in saved_files:
    size_kb = os.path.getsize(os.path.join(INDEX_DIR, f)) / 1024
    print(f"  {f:20s}  {size_kb:.1f} KB")

Index saved to: /home/hp/Desktop/chatbot/Customer-Feedback-Chabot/data/faiss_index
  index.faiss           334227.0 KB
  index.pkl             117292.7 KB


## 5. Smoke Test — Verify Retrieval

In [8]:
# Reload from disk to confirm persistence works
vs = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)

query = "unauthorized credit card charge"
results = vs.similarity_search(query, k=3)

print(f'Query: "{query}"\n')
for i, doc in enumerate(results, 1):
    print(f"[{i}] complaint_id={doc.metadata['complaint_id']}  "
          f"product={doc.metadata['product']}  "
          f"chunk={doc.metadata['chunk_index']}")
    print(f"    {doc.page_content[:150]}...\n")

Query: "unauthorized credit card charge"

[1] complaint_id=1390283  product=Credit card  chunk=6
    the company that put the unauthorized charge on my credit card...

[2] complaint_id=8223866  product=Credit card  chunk=2
    about the unauthorized charges made to my card there is no additional information to provide...

[3] complaint_id=8687697  product=Credit card  chunk=0
    credit card was activated by someone else with unauthorized charges along with open old disputes...

